# geometry.ipynb

Emotion-centroid geometry and CREMA-D intensity-transition analysis.

Set the representation roots and reference-index paths, then run the selected geometry sections.

Use the setup steps in the repository README before running this notebook.


In [ ]:
# =========================================================
# WEIGHTED SUBTLE CENTROIDS
# =========================================================
#
# Goal:
#
# Build annotator-weighted subtle emotion
# centroids.
#
# Example:
#
# {"joy":2, "pride":2, "approval":1}
#
# contributes:
#
# joy      -> weight 2/5
# pride    -> weight 2/5
# approval -> weight 1/5
#
# =========================================================

import ast
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================

TARGET_LAYER = "lm_2"

MODEL_FINEGRAINED = {

    "QWEN": {

        "pred_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_qwen/"
            "manipulated_probe_predictions.csv",

        "label_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_qwen/"
            "task3_results.csv"
    },

    "FLAMINGO": {

        "pred_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_flamingo/"
            "subtle_probe_predictions.csv",

        "label_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_flamingo/"
            "task3_results.csv"
    }
}

# =========================================================
# PARSERS
# =========================================================

def parse_labels(x):

    try:
        return ast.literal_eval(x)

    except:
        return []

def parse_label_counts(x):

    try:
        return ast.literal_eval(x)

    except:
        return {}

# =========================================================
# EMBEDDING LOADER
# =========================================================

def process_rep(x):

    if not torch.is_tensor(x):
        x = torch.tensor(x)

    x = x.float()

    # -----------------------------------------------------
    # pooling
    # -----------------------------------------------------

    if x.dim() == 3:
        x = x.mean(dim=1).squeeze(0)

    elif x.dim() == 2:
        x = x.squeeze(0)

    # -----------------------------------------------------
    # normalize
    # -----------------------------------------------------

    x = x / (x.norm() + 1e-8)

    return x.numpy()

def load_embedding(path, layer):

    try:

        rep = torch.load(
            path,
            map_location="cpu"
        )

        if layer not in rep:
            return None

        return process_rep(
            rep[layer]
        )

    except:
        return None

# =========================================================
# BUILD WEIGHTED CENTROIDS
# =========================================================

model_centroids = {}

for model_name, cfg in MODEL_FINEGRAINED.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    # =====================================================
    # LOAD
    # =====================================================

    pred_df = pd.read_csv(
        cfg["pred_csv"]
    )

    label_df = pd.read_csv(
        cfg["label_csv"]
    )

    # -----------------------------------------------------
    # clean paths
    # -----------------------------------------------------

    pred_df["rep_path"] = (
        pred_df["rep_path"]
        .astype(str)
        .str.strip()
    )

    label_df["rep_path"] = (
        label_df["rep_path"]
        .astype(str)
        .str.strip()
    )

    # =====================================================
    # MERGE
    # =====================================================

    df = pd.merge(

        pred_df,

        label_df[
            [
                "rep_path",
                "labels",
                "label_counts"
            ]
        ],

        on="rep_path",

        how="inner"
    )

    # =====================================================
    # FILTER LAYER
    # =====================================================

    df = df[
        df["layer"] == TARGET_LAYER
    ].copy()

    print("Merged rows:", len(df))

    # =====================================================
    # PARSE LABELS
    # =====================================================

    df["subtle_labels"] = (
        df["labels"]
        .apply(parse_labels)
    )

    df["label_count_dict"] = (
        df["label_counts"]
        .apply(parse_label_counts)
    )

    # =====================================================
    # COLLECT ALL SUBTLE LABELS
    # =====================================================

    all_subtle = set()

    for d in df["label_count_dict"]:

        for k in d.keys():
            all_subtle.add(k)

    print(
        "Unique subtle labels:",
        len(all_subtle)
    )

    # =====================================================
    # BUILD CENTROIDS
    # =====================================================

    centroids = {}

    for subtle in sorted(all_subtle):

        weighted_embs = []

        weights = []

        # -------------------------------------------------
        # iterate samples
        # -------------------------------------------------

        for _, row in tqdm(
            df.iterrows(),
            total=len(df),
            desc=subtle
        ):

            count_dict = row[
                "label_count_dict"
            ]

            # ---------------------------------------------
            # skip if subtle absent
            # ---------------------------------------------

            if subtle not in count_dict:
                continue

            # ---------------------------------------------
            # weight
            # ---------------------------------------------

            total_votes = sum(
                count_dict.values()
            )

            if total_votes == 0:
                continue

            weight = (
                count_dict[subtle]
                /
                total_votes
            )

            # ---------------------------------------------
            # embedding
            # ---------------------------------------------

            emb = load_embedding(
                row["rep_path"],
                TARGET_LAYER
            )

            if emb is None:
                continue

            weighted_embs.append(
                emb * weight
            )

            weights.append(weight)

        # -------------------------------------------------
        # finalize centroid
        # -------------------------------------------------

        if len(weighted_embs) == 0:
            continue

        centroid = (
            np.sum(
                weighted_embs,
                axis=0
            )
            /
            (
                np.sum(weights)
                + 1e-8
            )
        )

        # normalize
        centroid = (
            centroid
            /
            (
                np.linalg.norm(centroid)
                + 1e-8
            )
        )

        centroids[subtle] = centroid

        print(
            f"{subtle}: "
            f"{len(weights)} samples"
        )

    # =====================================================
    # SAVE MODEL CENTROIDS
    # =====================================================

    model_centroids[
        model_name
    ] = centroids

    print(
        f"\nBuilt {len(centroids)} centroids"
    )

In [ ]:
# =========================================================
# BLOCK 2
# BUILD CANONICAL COARSE CENTROIDS
# =========================================================
#
# Goal:
#
# Create CLEAN coarse centroids using
# only canonical/basic emotion samples.
#
# Why:
#
# subtle direction =
#   subtle centroid - coarse centroid
#
# becomes much cleaner if coarse centroids
# are NOT contaminated by subtle mixtures.
#
# =========================================================

import ast
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

# =========================================================
# BASIC EMOTIONS
# =========================================================

BASIC_MAP = {

    "anger": "ANG",

    "disgust": "DIS",

    "fear": "FEA",

    "joy": "HAP",

    "sadness": "SAD",

    "surprise": "SUR",

    "neutral": "NEU"
}

# =========================================================
# CONFIG
# =========================================================

TARGET_LAYER = "lm_2"

MODEL_FINEGRAINED = {

    "QWEN": {

        "pred_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_qwen/"
            "manipulated_probe_predictions.csv",

        "label_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_qwen/"
            "task3_results.csv"
    },

    "FLAMINGO": {

        "pred_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_flamingo/"
            "subtle_probe_predictions.csv",

        "label_csv":
            "/content/drive/MyDrive/subprobe/"
            "task3_full_run_flamingo/"
            "task3_results.csv"
    }
}

# =========================================================
# PARSERS
# =========================================================

def parse_labels(x):

    try:
        return ast.literal_eval(x)

    except:
        return []

def parse_label_counts(x):

    try:
        return ast.literal_eval(x)

    except:
        return {}

# =========================================================
# EMBEDDING
# =========================================================

def process_rep(x):

    if not torch.is_tensor(x):
        x = torch.tensor(x)

    x = x.float()

    # -----------------------------------------------------
    # pooling
    # -----------------------------------------------------

    if x.dim() == 3:
        x = x.mean(dim=1).squeeze(0)

    elif x.dim() == 2:
        x = x.squeeze(0)

    # -----------------------------------------------------
    # normalize
    # -----------------------------------------------------

    x = x / (x.norm() + 1e-8)

    return x.numpy()

def load_embedding(path, layer):

    try:

        rep = torch.load(
            path,
            map_location="cpu"
        )

        if layer not in rep:
            return None

        return process_rep(
            rep[layer]
        )

    except:
        return None

# =========================================================
# BUILD COARSE CENTROIDS
# =========================================================

model_coarse_centroids = {}

for model_name, cfg in MODEL_FINEGRAINED.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    # =====================================================
    # LOAD
    # =====================================================

    pred_df = pd.read_csv(
        cfg["pred_csv"]
    )

    label_df = pd.read_csv(
        cfg["label_csv"]
    )

    pred_df["rep_path"] = (
        pred_df["rep_path"]
        .astype(str)
        .str.strip()
    )

    label_df["rep_path"] = (
        label_df["rep_path"]
        .astype(str)
        .str.strip()
    )

    # =====================================================
    # MERGE
    # =====================================================

    df = pd.merge(

        pred_df,

        label_df[
            [
                "rep_path",
                "labels",
                "label_counts"
            ]
        ],

        on="rep_path",

        how="inner"
    )

    # =====================================================
    # FILTER LAYER
    # =====================================================

    df = df[
        df["layer"] == TARGET_LAYER
    ].copy()

    # =====================================================
    # PARSE
    # =====================================================

    df["subtle_labels"] = (
        df["labels"]
        .apply(parse_labels)
    )

    df["label_count_dict"] = (
        df["label_counts"]
        .apply(parse_label_counts)
    )

    # =====================================================
    # BUILD BASIC CENTROIDS
    # =====================================================

    coarse_centroids = {}

    for subtle_basic, coarse_name in BASIC_MAP.items():

        embs = []

        weights = []

        # -------------------------------------------------
        # iterate samples
        # -------------------------------------------------

        for _, row in tqdm(
            df.iterrows(),
            total=len(df),
            desc=coarse_name
        ):

            count_dict = row[
                "label_count_dict"
            ]

            # ---------------------------------------------
            # must contain target basic emotion
            # ---------------------------------------------

            if subtle_basic not in count_dict:
                continue

            # ---------------------------------------------
            # dominance filtering
            # keep only samples where canonical
            # emotion is strongest
            # ---------------------------------------------

            max_count = max(
                count_dict.values()
            )

            if (
                count_dict[subtle_basic]
                < max_count
            ):
                continue

            # ---------------------------------------------
            # compute weight
            # ---------------------------------------------

            total_votes = sum(
                count_dict.values()
            )

            weight = (
                count_dict[subtle_basic]
                /
                total_votes
            )

            # ---------------------------------------------
            # embedding
            # ---------------------------------------------

            emb = load_embedding(
                row["rep_path"],
                TARGET_LAYER
            )

            if emb is None:
                continue

            embs.append(
                emb * weight
            )

            weights.append(weight)

        # -------------------------------------------------
        # finalize centroid
        # -------------------------------------------------

        if len(embs) == 0:
            continue

        centroid = (

            np.sum(
                embs,
                axis=0
            )

            /

            (
                np.sum(weights)
                + 1e-8
            )
        )

        centroid = (
            centroid
            /
            (
                np.linalg.norm(centroid)
                + 1e-8
            )
        )

        coarse_centroids[
            coarse_name
        ] = centroid

        print(
            f"{coarse_name}: "
            f"{len(weights)} samples"
        )

    # =====================================================
    # SAVE
    # =====================================================

    model_coarse_centroids[
        model_name
    ] = coarse_centroids

    print(
        f"\nBuilt "
        f"{len(coarse_centroids)} "
        f"coarse centroids"
    )

In [ ]:
# =========================================================
# BLOCK 3
# LOAD INTENSITY TRANSITIONS
# =========================================================
#
# Goal:
#
# Build clean transition dataframe:
#
# MD -> HI
# LO -> MD
# LO -> HI
#
# using:
#
# source_rep = neutral_rep_path
# target_rep = rep_path
#
# =========================================================

import pandas as pd

# =========================================================
# CONFIG
# =========================================================

MODEL_TRANSITIONS = {

    "QWEN": {

        "csv":
            "/content/drive/MyDrive/probing/"
            "full_probe/albe/"
            "merged_results.csv"
    },

    "FLAMINGO": {

        "csv":
            "/content/drive/MyDrive/probing/"
            "full_probe_flamingo/abla/"
            "merged_results.csv"
    }
}

# =========================================================
# PARSER
# =========================================================

def parse_intensity_path(path):

    """
    Example:
    subset_audio\\1023_IEO_ANG_HI.wav
    """

    try:

        # ---------------------------------------------
        # windows path safe
        # ---------------------------------------------

        fname = str(path).split("\\")[-1]

        fname = fname.replace(
            ".wav",
            ""
        )

        parts = fname.split("_")

        return {

            "speaker":
                parts[0],

            "utterance":
                parts[1],

            "coarse":
                parts[2],

            "intensity":
                parts[3]
        }

    except Exception as e:

        print("Parse fail:", path)

        return None

# =========================================================
# BUILD TRANSITIONS
# =========================================================

all_rows = []

for model_name, cfg in MODEL_TRANSITIONS.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    df = pd.read_csv(
        cfg["csv"]
    )

    # =====================================================
    # INTENSITY ONLY
    # =====================================================

    df = df[
        df["task"] == "intensity"
    ].copy()

    print("Rows:", len(df))

    # =====================================================
    # BUILD TRANSITIONS
    # =====================================================

    for _, row in df.iterrows():

        src = parse_intensity_path(
            row["neutral_path"]
        )

        tgt = parse_intensity_path(
            row["manipulated_path"]
        )

        if src is None or tgt is None:
            continue

        # -------------------------------------------------
        # sanity checks
        # -------------------------------------------------

        if (
            src["speaker"]
            !=
            tgt["speaker"]
        ):
            continue

        if (
            src["utterance"]
            !=
            tgt["utterance"]
        ):
            continue

        if (
            src["coarse"]
            !=
            tgt["coarse"]
        ):
            continue

        # =================================================
        # SAVE
        # =================================================

        all_rows.append({

            # ---------------------------------------------
            # model
            # ---------------------------------------------

            "model":
                model_name,

            # ---------------------------------------------
            # metadata
            # ---------------------------------------------

            "speaker":
                src["speaker"],

            "utterance":
                src["utterance"],

            "coarse":
                src["coarse"],

            # ---------------------------------------------
            # transition
            # ---------------------------------------------

            "source_intensity":
                src["intensity"],

            "target_intensity":
                tgt["intensity"],

            "transition":
                (
                    f"{src['intensity']}"
                    f"_TO_"
                    f"{tgt['intensity']}"
                ),

            # ---------------------------------------------
            # representations
            # ---------------------------------------------

            "source_rep":
                row["neutral_rep_path"],

            "target_rep":
                row["rep_path"],

            # ---------------------------------------------
            # labels
            # ---------------------------------------------

            "source_label":
                row.get(
                    "neutral_label",
                    None
                ),

            "target_label":
                row.get(
                    "label",
                    None
                )
        })

# =========================================================
# FINAL DF
# =========================================================

transition_df = pd.DataFrame(all_rows)

print("\nFinal transitions:")
print(len(transition_df))

# =========================================================
# INSPECT
# =========================================================

print("\nTransition counts:\n")

print(
    transition_df["transition"]
    .value_counts()
)

print("\nCoarse counts:\n")

print(
    transition_df["coarse"]
    .value_counts()
)

print("\nExample:\n")

print(
    transition_df.head(10)
)

In [ ]:
# =========================================================
# BLOCK 4
# DIRECTIONAL FINE-GRAINED SHIFT ANALYSIS
# =========================================================
#
# Core idea:
#
# displacement =
#   target_embedding - source_embedding
#
# compare against:
#
# subtle_direction =
#   subtle_centroid - coarse_centroid
#
# using cosine similarity.
#
# =========================================================

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# COARSE -> SUBTLE MAP
# =========================================================

TARGET_SUBTLE = {

    # -----------------------------------------------------
    # ANGER
    # -----------------------------------------------------

    "ANG": [

        "anger",

        "annoyance",

        "disapproval"
    ],

    # -----------------------------------------------------
    # DISGUST
    # -----------------------------------------------------

    "DIS": [

        "disgust",

        "disapproval"
    ],

    # -----------------------------------------------------
    # FEAR
    # -----------------------------------------------------

    "FEA": [

        "fear",

        "nervousness"
    ],

    # -----------------------------------------------------
    # HAPPINESS
    # -----------------------------------------------------

    "HAP": [

        "joy",

        "amusement",

        "excitement",

        "pride",

        "optimism",

        "love",

        "admiration",

        "relief",

        "gratitude",

        "caring",

        "desire"
    ],

    # -----------------------------------------------------
    # NEUTRAL
    # -----------------------------------------------------

    "NEU": [

        "confusion",

        "curiosity",

        "realization",

        "surprise"
    ],

    # -----------------------------------------------------
    # SADNESS
    # -----------------------------------------------------

    "SAD": [

        "sadness",

        "disappointment",

        "remorse",

        "grief"
    ],

    # -----------------------------------------------------
    # SURPRISE
    # -----------------------------------------------------

    "SUR": [

        "surprise",

        "realization"
    ]
}

# =========================================================
# RUN ANALYSIS
# =========================================================

results = []

for model_name in transition_df["model"].unique():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    # =====================================================
    # MODEL DATA
    # =====================================================

    model_df = transition_df[
        transition_df["model"]
        == model_name
    ]

    # -----------------------------------------------------
    # model-specific centroids
    # -----------------------------------------------------

    subtle_centroids = (
        model_centroids[model_name]
    )

    coarse_centroids = (
        model_coarse_centroids[model_name]
    )

    # =====================================================
    # TRANSITIONS
    # =====================================================

    for _, row in tqdm(
        model_df.iterrows(),
        total=len(model_df)
    ):

        coarse = row["coarse"]

        # -------------------------------------------------
        # skip missing
        # -------------------------------------------------

        if coarse not in TARGET_SUBTLE:
            continue

        if coarse not in coarse_centroids:
            continue

        # =================================================
        # LOAD EMBEDDINGS
        # =================================================

        src_emb = load_embedding(
            row["source_rep"],
            TARGET_LAYER
        )

        tgt_emb = load_embedding(
            row["target_rep"],
            TARGET_LAYER
        )

        if src_emb is None or tgt_emb is None:
            continue

        # =================================================
        # REPRESENTATION DISPLACEMENT
        # =================================================

        delta = tgt_emb - src_emb

        delta_norm = np.linalg.norm(
            delta
        )

        if delta_norm < 1e-8:
            continue

        delta = (
            delta
            /
            delta_norm
        )

        # =================================================
        # SUBTLE DIRECTION ALIGNMENT
        # =================================================

        coarse_centroid = (
            coarse_centroids[coarse]
        )

        for subtle in TARGET_SUBTLE[coarse]:

            # ---------------------------------------------
            # subtle centroid exists?
            # ---------------------------------------------

            if subtle not in subtle_centroids:
                continue

            subtle_centroid = (
                subtle_centroids[subtle]
            )

            # ---------------------------------------------
            # direction:
            #
            # coarse -> subtle
            # ---------------------------------------------

            subtle_direction = (

                subtle_centroid
                -
                coarse_centroid
            )

            subtle_norm = np.linalg.norm(
                subtle_direction
            )

            if subtle_norm < 1e-8:
                continue

            subtle_direction = (

                subtle_direction
                /
                subtle_norm
            )

            # ---------------------------------------------
            # cosine alignment
            # ---------------------------------------------

            sim = cosine_similarity(

                delta.reshape(1, -1),

                subtle_direction.reshape(1, -1)

            )[0][0]

            # =================================================
            # SAVE
            # =================================================

            results.append({

                # ---------------------------------------------
                # model
                # ---------------------------------------------

                "model":
                    model_name,

                # ---------------------------------------------
                # metadata
                # ---------------------------------------------

                "speaker":
                    row["speaker"],

                "utterance":
                    row["utterance"],

                "coarse":
                    coarse,

                # ---------------------------------------------
                # transition
                # ---------------------------------------------

                "transition":
                    row["transition"],

                "source_intensity":
                    row["source_intensity"],

                "target_intensity":
                    row["target_intensity"],

                # ---------------------------------------------
                # subtle direction
                # ---------------------------------------------

                "subtle":
                    subtle,

                # ---------------------------------------------
                # metric
                # ---------------------------------------------

                "directional_similarity":
                    sim
            })

# =========================================================
# FINAL RESULTS
# =========================================================

shift_df = pd.DataFrame(results)

print("\nTotal results:")
print(len(shift_df))

# =========================================================
# SUMMARY
# =========================================================

summary = (

    shift_df

    .groupby([
        "model",
        "coarse",
        "transition",
        "subtle"
    ])

    ["directional_similarity"]

    .agg([
        "mean",
        "std",
        "count"
    ])

    .reset_index()

)

print("\nSummary:\n")

print(summary.head(100))

In [ ]:
# =========================================================
# BLOCK 5
# CANONICAL vs SUBTLE DIRECTION COMPARISON
# =========================================================
#
# Goal:
#
# Compare whether intensity shifts move toward:
#
# (1) canonical/basic emotion direction
# (2) subtle emotion direction
#
# Hypothesis:
#
# MD -> HI
#   aligns more with canonical direction
#
# MD -> LO
#   aligns more with subtle directions
#
# =========================================================

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# CANONICAL LABEL MAP
# =========================================================

CANONICAL_LABEL = {

    "ANG": "anger",

    "DIS": "disgust",

    "FEA": "fear",

    "HAP": "joy",

    "SAD": "sadness",

    "SUR": "surprise",

    "NEU": "neutral"
}

# =========================================================
# SUBTLE MAP
# =========================================================

TARGET_SUBTLE = {

    "ANG": [
        "annoyance",
        "disapproval"
    ],

    "DIS": [
        "disapproval"
    ],

    "FEA": [
        "nervousness"
    ],

    "HAP": [
        "amusement",
        "excitement",
        "pride",
        "optimism",
        "love",
        "admiration",
        "relief",
        "gratitude",
        "caring",
        "desire"
    ],

    "NEU": [
        "confusion",
        "curiosity",
        "realization"
    ],

    "SAD": [
        "disappointment",
        "remorse",
        "grief"
    ],

    "SUR": [
        "realization"
    ]
}

# =========================================================
# ANALYSIS
# =========================================================

comparison_results = []

for model_name in transition_df["model"].unique():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    model_df = transition_df[
        transition_df["model"]
        == model_name
    ]

    subtle_centroids = (
        model_centroids[model_name]
    )

    coarse_centroids = (
        model_coarse_centroids[model_name]
    )

    # =====================================================
    # TRANSITIONS
    # =====================================================

    for _, row in tqdm(
        model_df.iterrows(),
        total=len(model_df)
    ):

        coarse = row["coarse"]

        if coarse not in coarse_centroids:
            continue

        # -------------------------------------------------
        # canonical label
        # -------------------------------------------------

        canonical = CANONICAL_LABEL.get(
            coarse,
            None
        )

        if canonical is None:
            continue

        if canonical not in subtle_centroids:
            continue

        # =================================================
        # LOAD EMBEDDINGS
        # =================================================

        src_emb = load_embedding(
            row["source_rep"],
            TARGET_LAYER
        )

        tgt_emb = load_embedding(
            row["target_rep"],
            TARGET_LAYER
        )

        if src_emb is None or tgt_emb is None:
            continue

        # =================================================
        # DISPLACEMENT
        # =================================================

        delta = tgt_emb - src_emb

        delta_norm = np.linalg.norm(delta)

        if delta_norm < 1e-8:
            continue

        delta = delta / delta_norm

        # =================================================
        # CANONICAL DIRECTION
        # =================================================

        coarse_centroid = (
            coarse_centroids[coarse]
        )

        canonical_centroid = (
            subtle_centroids[canonical]
        )

        canonical_direction = (

            canonical_centroid
            -
            coarse_centroid
        )

        canonical_norm = np.linalg.norm(
            canonical_direction
        )

        if canonical_norm > 1e-8:

            canonical_direction = (

                canonical_direction
                /
                canonical_norm
            )

            canonical_sim = cosine_similarity(

                delta.reshape(1, -1),

                canonical_direction.reshape(1, -1)

            )[0][0]

        else:

            canonical_sim = np.nan

        # =================================================
        # SUBTLE ALIGNMENT
        # =================================================

        subtle_scores = []

        for subtle in TARGET_SUBTLE.get(
            coarse,
            []
        ):

            if subtle not in subtle_centroids:
                continue

            subtle_direction = (

                subtle_centroids[subtle]
                -
                coarse_centroid
            )

            subtle_norm = np.linalg.norm(
                subtle_direction
            )

            if subtle_norm < 1e-8:
                continue

            subtle_direction = (

                subtle_direction
                /
                subtle_norm
            )

            sim = cosine_similarity(

                delta.reshape(1, -1),

                subtle_direction.reshape(1, -1)

            )[0][0]

            subtle_scores.append(sim)

        # -------------------------------------------------
        # average subtle alignment
        # -------------------------------------------------

        if len(subtle_scores) == 0:
            continue

        avg_subtle_sim = np.mean(
            subtle_scores
        )

        # =================================================
        # SAVE
        # =================================================

        comparison_results.append({

            "model":
                model_name,

            "coarse":
                coarse,

            "transition":
                row["transition"],

            # ---------------------------------------------
            # scores
            # ---------------------------------------------

            "canonical_alignment":
                canonical_sim,

            "subtle_alignment":
                avg_subtle_sim,

            # ---------------------------------------------
            # difference
            # positive:
            # more canonical
            #
            # negative:
            # more subtle
            # ---------------------------------------------

            "canonical_minus_subtle":
                (
                    canonical_sim
                    -
                    avg_subtle_sim
                )
        })

# =========================================================
# FINAL DF
# =========================================================

comparison_df = pd.DataFrame(
    comparison_results
)

print("\nTotal rows:")
print(len(comparison_df))

# =========================================================
# SUMMARY
# =========================================================

summary = (

    comparison_df

    .groupby([
        "model",
        "coarse",
        "transition"
    ])

    [
        [
            "canonical_alignment",
            "subtle_alignment",
            "canonical_minus_subtle"
        ]
    ]

    .mean()

    .reset_index()

)

print("\nSummary:\n")

print(summary)

In [ ]:
shift_df.to_csv(
    "fine_grained_shift_results.csv",
    index=False
)

In [ ]:
summary.to_csv(
    "fine_grained_shift_summary.csv",
    index=False
)

In [ ]:
comparison_df.to_csv(
    "canonical_subtle_comparison.csv",
    index=False
)

In [ ]:
comparison_summary = (
    comparison_df
    .groupby([
        "model",
        "coarse",
        "transition"
    ])
    [[
        "canonical_alignment",
        "subtle_alignment",
        "canonical_minus_subtle"
    ]]
    .agg([
        "mean",
        "std"
    ])
    .reset_index()
)

comparison_summary.to_csv(
    "canonical_subtle_summary.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plot_df = (
    comparison_df
    .groupby([
        "model",
        "transition"
    ])
    ["subtle_alignment"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7,5))

for model in plot_df["model"].unique():

    sub = plot_df[
        plot_df["model"] == model
    ]

    ax.plot(
        sub["transition"],
        sub["subtle_alignment"],
        marker="o",
        label=model
    )

ax.axhline(0, linestyle="--")

ax.set_ylabel(
    "Mean subtle-direction alignment"
)

ax.set_xlabel(
    "Intensity transition"
)

ax.set_title(
    "Speech intensity perturbation and subtle-emotion alignment"
)

ax.legend()

plt.tight_layout()

plt.savefig(
    "subtle_alignment_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================================================
# AGGREGATE
# =========================================================

plot_df = (
    comparison_df
    .groupby([
        "model",
        "coarse",
        "transition"
    ])
    ["subtle_alignment"]
    .mean()
    .reset_index()
)

coarse_list = sorted(
    plot_df["coarse"].unique()
)

# =========================================================
# SIDE-BY-SIDE FIGURE
# =========================================================

models = plot_df["model"].unique()

fig, axes = plt.subplots(
    1,
    len(models),
    figsize=(14, 5),
    sharey=True
)

# ---------------------------------------------------------
# handle single model case
# ---------------------------------------------------------

if len(models) == 1:
    axes = [axes]

# =========================================================
# PLOT EACH MODEL
# =========================================================

for ax, model in zip(axes, models):

    sub = plot_df[
        plot_df["model"] == model
    ]

    x = np.arange(len(coarse_list))

    width = 0.35

    vals_hi = []
    vals_lo = []

    for coarse in coarse_list:

        hi = sub[
            (sub["coarse"] == coarse)
            &
            (sub["transition"] == "MD_TO_HI")
        ]["subtle_alignment"]

        lo = sub[
            (sub["coarse"] == coarse)
            &
            (sub["transition"] == "MD_TO_LO")
        ]["subtle_alignment"]

        vals_hi.append(
            hi.iloc[0] if len(hi) else 0
        )

        vals_lo.append(
            lo.iloc[0] if len(lo) else 0
        )

    # -----------------------------------------------------
    # bars
    # -----------------------------------------------------

    ax.bar(
        x - width/2,
        vals_hi,
        width,
        label="MD→HI"
    )

    ax.bar(
        x + width/2,
        vals_lo,
        width,
        label="MD→LO"
    )

    # -----------------------------------------------------
    # formatting
    # -----------------------------------------------------

    ax.axhline(
        0,
        linestyle="--"
    )

    ax.set_xticks(x)

    ax.set_xticklabels(coarse_list)

    ax.set_title(model)

    ax.set_xlabel(
        "Coarse emotion"
    )

# =========================================================
# SHARED LABELS
# =========================================================

axes[0].set_ylabel(
    "Subtle-direction alignment"
)

# =========================================================
# SHARED LEGEND
# =========================================================

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    bbox_to_anchor=(0.5, 1.02)
)

# =========================================================
# TITLE
# =========================================================

fig.suptitle(
    "Intensity-induced representational shift toward subtle emotional regions",
    y=1.08
)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    "combined_coarse_shift_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# HEATMAP:
# WHICH SUBTLE EMOTION DOES THE SHIFT MOVE TOWARD?
# =========================================================

import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# AGGREGATE
# =========================================================

heatmap_df = (

    shift_df

    .groupby([
        "model",
        "transition",
        "coarse",
        "subtle"
    ])

    ["directional_similarity"]

    .mean()

    .reset_index()
)

print(heatmap_df.head())

# =========================================================
# PLOT
# =========================================================

for model_name in heatmap_df["model"].unique():

    for transition in heatmap_df["transition"].unique():

        sub = heatmap_df[

            (heatmap_df["model"] == model_name)
            &
            (heatmap_df["transition"] == transition)

        ]

        # -------------------------------------------------
        # pivot table
        # rows = coarse
        # cols = subtle
        # -------------------------------------------------

        pivot = sub.pivot(

            index="coarse",
            columns="subtle",
            values="directional_similarity"
        )

        # -------------------------------------------------
        # plot
        # -------------------------------------------------

        fig, ax = plt.subplots(
            figsize=(12, 4)
        )

        im = ax.imshow(
            pivot.values,
            aspect="auto"
        )

        # -------------------------------------------------
        # ticks
        # -------------------------------------------------

        ax.set_xticks(
            range(len(pivot.columns))
        )

        ax.set_xticklabels(
            pivot.columns,
            rotation=45,
            ha="right"
        )

        ax.set_yticks(
            range(len(pivot.index))
        )

        ax.set_yticklabels(
            pivot.index
        )

        # -------------------------------------------------
        # values
        # -------------------------------------------------

        for i in range(pivot.shape[0]):

            for j in range(pivot.shape[1]):

                val = pivot.values[i, j]

                if pd.isna(val):
                    continue

                ax.text(
                    j,
                    i,
                    f"{val:.2f}",
                    ha="center",
                    va="center",
                    fontsize=8
                )

        # -------------------------------------------------
        # title
        # -------------------------------------------------

        ax.set_title(
            f"{model_name} | {transition}\n"
            "Subtle-emotion directional alignment"
        )

        # -------------------------------------------------
        # colorbar
        # -------------------------------------------------

        cbar = plt.colorbar(im)

        cbar.set_label(
            "Directional similarity"
        )

        plt.tight_layout()

        # -------------------------------------------------
        # save
        # -------------------------------------------------

        plt.savefig(
            f"{model_name}_{transition}_heatmap.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()

In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

TARGET_LAYER = "lm_2"

MODELS = {
    "QWEN": {
        "csv": "/content/drive/MyDrive/probe/probe_representations_2/probe_rep_index.csv"
    },
    "FLAMINGO": {
        "csv": "/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_rep_index.csv"
    }
}

def process_rep(x):

    if not torch.is_tensor(x):
        x = torch.tensor(x)

    x = x.float()

    if x.dim() == 3:
        x = x.mean(dim=1).squeeze(0)
    elif x.dim() == 2:
        x = x.squeeze(0)

    x = x / (x.norm() + 1e-8)

    return x.numpy()


def load_embedding(path, layer):

    rep = torch.load(path, map_location="cpu")

    if layer not in rep:
        return None

    return process_rep(rep[layer])


model_centroids = {}

for model_name, cfg in MODELS.items():

    print("="*60)
    print(model_name)
    print("="*60)

    df = pd.read_csv(cfg["csv"])

    df["rep_path"] = df["rep_path"].astype(str).str.strip()

    centroids = {}

    for emotion in sorted(df["emotion"].unique()):

        subset = df[df["emotion"] == emotion]

        embs = []

        for _, row in tqdm(subset.iterrows(),
                           total=len(subset),
                           desc=emotion):

            emb = load_embedding(
                row["rep_path"],
                TARGET_LAYER
            )

            if emb is not None:
                embs.append(emb)

        embs = np.stack(embs)

        centroid = embs.mean(axis=0)
        centroid /= np.linalg.norm(centroid) + 1e-8

        centroids[emotion] = centroid

        print(f"{emotion}: {len(embs)} samples")

    model_centroids[model_name] = centroids

In [ ]:
# =========================================================
# BLOCK 5 (v2)
# CENTROID DISTANCE ANALYSIS
# =========================================================
#
# Question:
#
# Does increasing intensity move representations
# closer to the canonical emotion centroid?
#
# MD -> HI : should become closer
# MD -> LO : should become further
#
# =========================================================

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

comparison_results = []

for model_name in transition_df["model"].unique():

    print("\n" + "="*60)
    print(model_name)
    print("="*60)

    model_df = transition_df[
        transition_df["model"] == model_name
    ]

    centroids = model_centroids[model_name]

    for _, row in tqdm(
        model_df.iterrows(),
        total=len(model_df)
    ):

        emotion = row["coarse"]

        if emotion not in centroids:
            continue

        centroid = centroids[emotion]

        # ----------------------------------------
        # load embeddings
        # ----------------------------------------

        src_emb = load_embedding(
            row["source_rep"],
            TARGET_LAYER
        )

        tgt_emb = load_embedding(
            row["target_rep"],
            TARGET_LAYER
        )

        if src_emb is None or tgt_emb is None:
            continue

        # ----------------------------------------
        # cosine similarity to centroid
        # ----------------------------------------

        src_sim = cosine_similarity(
            src_emb.reshape(1,-1),
            centroid.reshape(1,-1)
        )[0][0]

        tgt_sim = cosine_similarity(
            tgt_emb.reshape(1,-1),
            centroid.reshape(1,-1)
        )[0][0]

        comparison_results.append({

            "model": model_name,

            "emotion": emotion,

            "transition": row["transition"],

            "source_similarity": src_sim,

            "target_similarity": tgt_sim,

            "similarity_change":
                tgt_sim - src_sim

        })

# =========================================================
# FINAL DATAFRAME
# =========================================================

comparison_df = pd.DataFrame(
    comparison_results
)

print("Total rows:", len(comparison_df))

# =========================================================
# SUMMARY
# =========================================================

summary = (

    comparison_df

    .groupby([
        "model",
        "transition"
    ])

    [
        [
            "source_similarity",
            "target_similarity",
            "similarity_change"
        ]
    ]

    .mean()

    .reset_index()

)

print(summary)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================================================
# AGGREGATE
# =========================================================

plot_df = (
    comparison_df
    .groupby([
        "model",
        "emotion",
        "transition"
    ])["similarity_change"]
    .mean()
    .reset_index()
)

emotion_list = sorted(plot_df["emotion"].unique())

# =========================================================
# FIGURE
# =========================================================

models = plot_df["model"].unique()

fig, axes = plt.subplots(
    1,
    len(models),
    figsize=(14,5),
    sharey=False
)

if len(models) == 1:
    axes = [axes]

# =========================================================
# PLOT
# =========================================================

for ax, model in zip(axes, models):

    sub = plot_df[
        plot_df["model"] == model
    ]

    x = np.arange(len(emotion_list))
    width = 0.35

    vals_hi = []
    vals_lo = []

    for emotion in emotion_list:

        hi = sub[
            (sub["emotion"] == emotion)
            &
            (sub["transition"] == "MD_TO_HI")
        ]["similarity_change"]

        lo = sub[
            (sub["emotion"] == emotion)
            &
            (sub["transition"] == "MD_TO_LO")
        ]["similarity_change"]

        vals_hi.append(
            hi.iloc[0] if len(hi) else np.nan
        )

        vals_lo.append(
            lo.iloc[0] if len(lo) else np.nan
        )

    ax.bar(
        x - width/2,
        vals_hi,
        width,
        label="MD→HI"
    )

    ax.bar(
        x + width/2,
        vals_lo,
        width,
        label="MD→LO"
    )

    ax.axhline(
        0,
        linestyle="--"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(emotion_list)

    ax.set_title(model)

    ax.set_xlabel(
        "Emotion"
    )

# =========================================================
# LABELS
# =========================================================

axes[0].set_ylabel(
    "Change in cosine similarity\n(to emotion centroid)"
)

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    bbox_to_anchor=(0.5,1.02)
)

fig.suptitle(
    "Effect of intensity manipulation on similarity to emotion centroids",
    y=1.08
)

plt.tight_layout()

plt.savefig(
    "centroid_similarity_shift.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:


table = (
    comparison_df
    .groupby([
        "model",
        "emotion",
        "transition"
    ])["similarity_change"]
    .mean()
    .unstack("transition")
    .reset_index()
)

table = table.rename(columns={
    "MD_TO_HI": "MD→HI",
    "MD_TO_LO": "MD→LO"
})

table = table.round(4)

print(table)